# Tabular Data Processing with Prior Labs MCP

In this recipe, we scrape hotel listings from the web, structure them into a table using an LLM, and then use a Haystack `Agent` equipped with [Prior Labs' TabPFN](https://priorlabs.ai/) tools to predict and fill in missing attributes.

**Services used:**
- [Firecrawl](https://www.firecrawl.dev/) — web scraping
- [Anthropic Claude](https://www.anthropic.com/) — LLM for extraction and agent reasoning
- [Prior Labs MCP](https://priorlabs.ai/) — tabular ML predictions via MCP tools

## Install dependencies

In [ ]:
!pip install -q haystack-ai anthropic-haystack mcp-haystack trafilatura firecrawl-haystack

## Set up API keys

You'll need three API keys:
- **Anthropic API key** — get one at [console.anthropic.com](https://console.anthropic.com/)
- **Firecrawl API key** — get one at [firecrawl.dev](https://www.firecrawl.dev/)
- **Prior Labs MCP token** — get one at [ux.priorlabs.ai](https://ux.priorlabs.ai/)

In [ ]:
import os
from getpass import getpass

if "ANTHROPIC_API_KEY" not in os.environ:
    os.environ["ANTHROPIC_API_KEY"] = getpass("Enter your Anthropic API key: ")
if "FIRECRAWL_API_KEY" not in os.environ:
    os.environ["FIRECRAWL_API_KEY"] = getpass("Enter your Firecrawl API key: ")
if "PRIOR_LABS_MCP_TOKEN" not in os.environ:
    os.environ["PRIOR_LABS_MCP_TOKEN"] = getpass("Enter your Prior Labs MCP token: ")

## Step 1: Scrape and structure hotel listings

We build a pipeline that fetches hotel search results with [`FirecrawlCrawler`](https://haystack.deepset.ai/integrations/firecrawl) and passes the raw text to Claude, which extracts the data into a structured Markdown table. Any missing fields are left blank - we'll fill those in with ML predictions in the next step.

In [16]:
from haystack_integrations.components.generators.anthropic import AnthropicChatGenerator
from haystack_integrations.components.fetchers.firecrawl import FirecrawlCrawler
from haystack.components.builders import ChatPromptBuilder
from haystack.dataclasses import ChatMessage
from haystack import Pipeline


pipeline = Pipeline()
pipeline.add_component("fetcher", FirecrawlCrawler())
pipeline.add_component(
    "prompt_builder", 
    ChatPromptBuilder(
        template=[
            ChatMessage.from_system(
                "You are a hotel booking assistant. Extract the relevant information from the text and summarize it in a concise manner, as a Markdown table with the following columns: \n- Hotel name\n- Location\n- Distance from the target\n- Price per night\n- Room type\n- User rating\n- Location rating\n- Number of reviews\n- Number and types of beds\n- Amenities\n- Room size\n- Whether breakfast is included\n- Public transport availability\n\nIf any of the information is missing, leave the corresponding cell empty. Always return all the hotels from the input, even if some of the information is missing."
            ),
            ChatMessage.from_user(
                "{% for doc in documents %}{{ doc.content }}\n{% endfor %}"
            ),
        ],
        required_variables=["documents"],
    ),
)
pipeline.add_component(
    "summarizer", 
    AnthropicChatGenerator(
        model="claude-opus-4-6",
        generation_kwargs={
            "max_tokens": 10000,
        }
    ),
)

pipeline.connect("fetcher.documents", "prompt_builder.documents")
pipeline.connect("prompt_builder.prompt", "summarizer.messages")

🚅 Components
  - fetcher: FirecrawlCrawler
  - prompt_builder: ChatPromptBuilder
  - summarizer: AnthropicChatGenerator
🛤️ Connections
  - fetcher.documents -> prompt_builder.documents (list[Document])
  - prompt_builder.prompt -> summarizer.messages (list[ChatMessage])

In [17]:
output = pipeline.run({
    "fetcher": {
        "urls": [
            "https://www.booking.com/searchresults.html?ss=Berlin%2C+Berlin+Federal+State%2C+Germany&efdco=1&label=gen173nr-10CAEoggI46AdIM1gEaLYBiAEBmAEzuAEHyAEM2AED6AEB-AEBiAIBqAIBuAKQgrTOBsACAdICJDllMDgyNTA1LTdlYmMtNDJkNi04NzFmLWUyYTRkNWM1NjM3ZtgCAeACAQ&aid=304142&lang=en-us&sb=1&src_elem=sb&src=index&dest_id=-1746443&dest_type=city&ac_position=0&ac_click_type=b&ac_langcode=xu&ac_suggestion_list_length=5&search_selected=true&search_pageview_id=0da2508885520c7d&ac_meta=GhAwZGEyNTA4ODg1NTIwYzdkIAAoATICeHU6BmJlcmxpbg%3D%3D&checkin=2026-05-05&checkout=2026-05-08&group_adults=1&no_rooms=1&group_children=0",
        ]
    }
})

In [18]:
from IPython.display import Markdown, display

display(Markdown(output["summarizer"]["replies"][-1].text))

| Hotel Name | Location | Distance from Downtown | Price per Night | Room Type | User Rating | Location Rating | Number of Reviews | Beds | Amenities | Room Size | Breakfast Included | Public Transport |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| oompH Berlin 3BR Apt with Light Tones and Artwork | Friedrichshain-Kreuzberg, Berlin | 3.9 miles | $210 | Three-Bedroom Superior Apartment | 9.3 Wonderful | 9.3 | 25 | 5 beds (1 full, 1 king, 2 sofa beds, 1 queen) | Free Wifi | 1,012 ft² | | Subway Access |
| Hotel Berlin, Berlin, a member of Radisson Individuals | Mitte, Berlin | 1.3 miles | $184 | Cozy Small Room | 8.2 Very Good | | 19,637 | 1 queen bed | Bar, Fitness center, Sustainability certification | | | Subway Access |
| Lux 11 Berlin-Mitte | Mitte, Berlin | 1.5 miles | $218 | Deluxe Room | 8.7 Excellent | 9.5 | 2,073 | 1 king bed | Bar, Restaurant, Free Wifi | | | Subway Access |
| Premier Inn Berlin Airport | Treptow-Köpenick, Berlin | 10.8 miles | $83 | Quadruple Room | 8.6 Excellent | | 10,046 | 2 beds (1 twin, 1 full) | Restaurant, Free Wifi, 24-hour front desk | | | |
| Steigenberger Hotel Am Kanzleramt | Mitte, Berlin | 0.6 miles | $249 | Superior Plus Room | 8.8 Excellent | 9.5 | 7,654 | 1 double or 2 twins | Bar, Spa, Sustainability certification | | | Subway Access |
| Numa Berlin Boxer | Friedrichshain-Kreuzberg, Berlin | 3.8 miles | $149 | Large Room - Accessible | 8.1 Very Good | | 1,577 | 1 king bed | Fitness center, Free Wifi | | | |
| Hollywood Media Hotel am Kurfürstendamm | Charlottenburg-Wilmersdorf, Berlin | 2.5 miles | $158 | Standard Double or Twin Room | 8.5 Very Good | 9.4 | 7,307 | 1 double or 2 twins | Bar, Fitness center, Sustainability certification | | | Subway Access |
| art'otel berlin mitte, Powered by Radisson Hotels | Mitte, Berlin | 1.4 miles | $233 | Art Room courtyard view | 8.9 Excellent | 9.3 | 3,347 | 1 double or 2 twins | Bar, Spa, Sustainability certification | | | Subway Access |
| Wilmina Hotel | Charlottenburg-Wilmersdorf, Berlin | 3.4 miles | $265 | Cozy Double Room | 9.4 Wonderful | | 1,189 | 1 queen bed | Bar, Fitness center, Sustainability certification | | | Subway Access |
| The Mandala Suites | Mitte, Berlin | 0.6 miles | $284 | Management Suite | 9.0 Wonderful | 9.6 | 2,225 | 1 king bed | Spa, Fitness center, Sustainability certification | 732 ft² | | Subway Access |
| Townhouse Berlin | Charlottenburg-Wilmersdorf, Berlin | 2.3 miles | $187 | City Studio | 8.8 Excellent | 9.3 | 1,216 | 1 queen bed | Fitness center, Free Wifi, Sustainability certification | 280 ft² | | Subway Access |
| Numa Berlin Torstrasse | Mitte, Berlin | 1.6 miles | $189 | Medium Room | 8.4 Very Good | 9.4 | 2,737 | 1 queen bed | Fitness center, Free Wifi | | | Subway Access |
| 25hours Hotel Bikini Berlin | Charlottenburg-Wilmersdorf, Berlin | 1.8 miles | $252 | Medium Urban Twin | 8.7 Excellent | 9.5 | 2,177 | 2 twin beds | Bar, Sauna, Sustainability certification | | | Subway Access |
| NH Collection Berlin Mitte am Checkpoint Charlie | Mitte, Berlin | 0.6 miles | $258 | Premium Double or Twin Room | 8.5 Very Good | 9.4 | 8,085 | Multiple bed types | Bar, Fitness center, Sustainability certification | | | Subway Access |
| The Social Hub Berlin Alexanderplatz | Mitte, Berlin | 1.7 miles | $152 | Standard Queen Room | 8.6 Excellent | 9.3 | 9,278 | 1 queen bed | Bar, Fitness center, Restaurant | | | Subway Access |
| Hotel Augusta Am Kurfürstendamm | Charlottenburg-Wilmersdorf, Berlin | 2.3 miles | $116 | Classic Room | 8.1 Very Good | 9.4 | 2,951 | 1 full bed | Hot tub/Jacuzzi, Free Wifi, Baggage storage | | | Subway Access |
| Hotel ROMY by AMANO | Mitte, Berlin | 0.8 miles | $197 | Standard Double Room | 8.5 Very Good | 9.5 | 5,969 | 1 queen bed | Bar, Restaurant, Sustainability certification | | | Subway Access |
| Living Hotel Berlin Mitte | Mitte, Berlin | 1.4 miles | $181 | Business Double Apartment | 8.6 Excellent | | 1,485 | 1 queen bed | Bar, Sauna, Sustainability certification | 248 ft² | | Subway Access |
| Numa Berlin Arc | Friedrichshain-Kreuzberg, Berlin | 1 mile | $130 | Twin Room | 8.3 Very Good | | 5,269 | 2 twin beds | Fitness center, Free Wifi, Baggage storage | | | Subway Access |
| BENSIMON apartments Prenzlauer Berg | Pankow, Berlin | 2 miles | $152 | Studio Apartment with Garden View | 9.2 Wonderful | 9.5 | 597 | 2 beds (1 sofa bed, 1 queen) | Free Wifi, Baggage storage | 431 ft² | | Subway Access |
| Hotel Sachsenhof | Tempelhof-Schöneberg, Berlin | 1.6 miles | $134 | Triple Room | 8.5 Very Good | 9.3 | 5,242 | 2 beds (1 twin, 1 queen) | Free Wifi, 24-hour front desk, Baggage storage | | | Subway Access |
| InterContinental Berlin by IHG | Mitte, Berlin | 1.5 miles | $170 | Classic Room | 8.8 Excellent | | 4,778 | 1 double or 2 twins | Bar, Spa, Sustainability certification | | | |
| martas Hotel Berlin-Mitte | Mitte, Berlin | 1 mile | $180 | Double Room | 8.8 Excellent | 9.6 | 2,619 | 1 queen bed | Free Wifi, Baggage storage | | | Subway Access |
| Orania.Berlin | Friedrichshain-Kreuzberg, Berlin | 1.9 miles | $328 | Double Room (Orania.25) | 9.4 Wonderful | | 536 | 1 queen bed | Bar, Fitness center, Restaurant | | | Subway Access |
| Pestana Berlin Tiergarten | Mitte, Berlin | 1.3 miles | $184 | Deluxe Premium Room | 8.4 Very Good | | 5,473 | 1 king bed | Bar, Fitness center, Sauna | | | |
| Park Plaza Berlin | Charlottenburg-Wilmersdorf, Berlin | 2.5 miles | $239 | Superior Double Room | 8.3 Very Good | | 5,253 | 1 queen bed | Bar, Fitness center, Sustainability certification | | | Subway Access |
| Holiday Inn Berlin City East Side by IHG | Friedrichshain-Kreuzberg, Berlin | 3 miles | $199 | Standard Twin Room with Courtyard View | 8.7 Excellent | 9.5 | 5,239 | 2 twin beds | Bar, Fitness center, Restaurant | | | Subway Access |
| Myer's Hotel Berlin | Pankow, Berlin | 2 miles | $256 | Comfort Double or Twin Room | 9.0 Wonderful | | 997 | 1 double or 2 twins | Bar, Spa, Fitness center | | Yes | Subway Access |
| Dorint Kurfürstendamm Berlin | Charlottenburg-Wilmersdorf, Berlin | 2.1 miles | $259 | Comfort Room with Kingsize Bed | 8.8 Excellent | 9.5 | 4,691 | 1 king bed | Bar, Fitness center, Sustainability certification | | | Subway Access |
| The Ritz-Carlton, Berlin | Mitte, Berlin | 0.4 miles | $433 | Deluxe King Room | 9.3 Wonderful | 9.6 | 1,097 | 1 king bed | Bar, Fitness center, Sustainability certification | | | Subway Access |
| Radisson RED Berlin Kudamm | Charlottenburg-Wilmersdorf, Berlin | 2.2 miles | $242 | Standard Single Room | 8.6 Excellent | | 2,427 | 1 twin bed | Fitness center, Restaurant, Sustainability certification | | | Subway Access |
| IntercityHotel Berlin Hauptbahnhof | Mitte, Berlin | 0.7 miles | $244 | Business Double Room | 8.0 Very Good | 9.4 | 10,484 | 1 queen bed | Bar, Free Wifi, Sustainability certification | | | Subway Access |
| The Mandala Berlin, a Member of Design Hotels | Mitte, Berlin | 0.5 miles | $313 | Garden Studio with Balcony | 9.1 Wonderful | 9.5 | 1,746 | 1 queen bed | Bar, Spa, Fitness center | | | Subway Access |
| Telegraphenamt | Mitte, Berlin | 0.9 miles | $454 | Deluxe Double Room | 8.8 Excellent | 9.7 | 720 | 1 king bed | Bar, Fitness center, Sustainability certification | | | |
| Arbio I Luxury Apartments in East Side Gallery | Friedrichshain-Kreuzberg, Berlin | 2.8 miles | $161 | Comfort Studio | 8.5 Very Good | 9.4 | 1,004 | 2 beds (1 full, 1 sofa bed) | Free Wifi, Baggage storage | 344 ft² | | Subway Access |
| Hotel Gat Point Charlie | Mitte, Berlin | 0.7 miles | $128 | Standard Double Room | 8.5 Very Good | 9.5 | 7,548 | 1 double or 2 twins | Bar, Free Wifi, Sustainability certification | | | Subway Access |
| Berlin Marriott Hotel | Mitte, Berlin | 0.4 miles | $302 | Deluxe Room, 1 King | 8.6 Excellent | 9.5 | 3,141 | 1 king bed | Bar, Fitness center, Sustainability certification | | | Subway Access |
| Hotel Zoo Berlin | Charlottenburg-Wilmersdorf, Berlin | 2.2 miles | $238 | Superior Room | 8.8 Excellent | 9.6 | 3,205 | 1 double or 2 twins | Bar, Fitness center, Sustainability certification | | | Subway Access |
| Hotel AMANO Grand Central | Mitte, Berlin | 0.9 miles | $229 | COMFY Plus | 8.2 Very Good | 9.3 | 10,475 | 1 double or 2 twins | Bar, Restaurant, Sustainability certification | | | Subway Access |
| Art Apartments | Mitte, Berlin | 1.1 miles | $160 | Comfort Triple Room with Shower | 7.2 Good | | 1,724 | 3 king beds | Free Wifi | | | Subway Access |
| Hotel AMO by AMANO Friedrichstraße | Mitte, Berlin | 0.8 miles | $215 | COMFY Room | 8.3 Very Good | 9.4 | 4,562 | 1 double or 2 twins | Bar, Restaurant, Sustainability certification | | | Subway Access |
| Melarose Feng Shui Hotel | Pankow, Berlin | 2.6 miles | $146 | Single Room | 8.3 Very Good | | 1,478 | 1 twin bed | Free Wifi, Baggage storage | | | |
| Hotel Nikolai Residence | Mitte, Berlin | 1.3 miles | $183 | Double Room | 8.9 Excellent | 9.7 | 2,562 | 1 double or 2 twins | Free Wifi, Baggage storage | | | Subway Access |
| The Hoxton, Berlin | Charlottenburg-Wilmersdorf, Berlin | 2.3 miles | $233 | Standard Twin Room | 8.6 Excellent | | 2,385 | 2 twin beds | Bar, Free Wifi, Sustainability certification | | | Subway Access |
| Motel One Berlin-Alexanderplatz | Mitte, Berlin | 1.5 miles | $158 | Standard Double Room | 8.7 Excellent | 9.6 | 16,191 | 1 queen bed | Bar, Free Wifi, Sustainability certification | | | Subway Access |
| Scandic Berlin Potsdamer Platz | Mitte, Berlin | 0.8 miles | $198 | Standard Triple Room | 8.4 Very Good | 9.3 | 11,774 | 2 beds (1 sofa bed, 1 queen) | Bar, Fitness center, Sustainability certification | | | Subway Access |
| Lindner Hotel Berlin Ku'damm, part of JdV by Hyatt | Charlottenburg-Wilmersdorf, Berlin | 2.2 miles | $182 | Queen Room - High Floor | 8.4 Very Good | 9.6 | 2,175 | 1 queen bed | Restaurant, Free Wifi, 24-hour front desk | | | Subway Access |
| Hotel MANI by AMANO | Mitte, Berlin | 1.3 miles | $179 | Double Room | 8.3 Very Good | 9.3 | 2,430 | 1 double or 2 twins | Bar, Restaurant, Sustainability certification | | | Subway Access |
| Hotel Q! Berlin | Charlottenburg-Wilmersdorf, Berlin | 2.5 miles | $171 | Single Room | 8.7 Excellent | 9.5 | 1,196 | 1 twin bed | Fitness center, Sauna, Free Wifi | | | Subway Access |
| Hotel Adlon Kempinski Berlin | Mitte, Berlin | 0.1 miles | $417 | Executive Room King | 9.1 Wonderful | 9.8 | 3,891 | 1 king bed | Bar, Fitness center, Hot tub/Jacuzzi | | | Subway Access |
| Maritim proArte Hotel Berlin | Mitte, Berlin | 0.5 miles | $291 | Classic Double Room | 8.5 Very Good | 9.7 | 11,379 | 1 double or 2 twins | Bar, Spa, Sustainability certification | | | Subway Access |
| aletto Hotel Potsdamer Platz | Friedrichshain-Kreuzberg, Berlin | 1.1 miles | $121 | Double or Twin Room | 8.8 Excellent | | 8,842 | 2 twin beds | Bar, Restaurant, Sustainability certification | | | Subway Access |
| Holiday Inn Express Berlin City Centre by IHG | Friedrichshain-Kreuzberg, Berlin | 1.1 miles | $173 | Standard Room | 8.5 Very Good | | 9,791 | 1 double or 2 twins | Bar, Free Wifi, Sustainability certification | | Yes | Subway Access |
| DoubleTree by Hilton Berlin Ku'damm | Charlottenburg-Wilmersdorf, Berlin | 2 miles | $252 | Twin Room | 8.7 Excellent | 9.4 | 3,942 | 2 twin beds | Bar, Fitness center, Sustainability certification | | | Subway Access |
| Industriepalast Berlin | Friedrichshain-Kreuzberg, Berlin | 3.1 miles | $24 | Single Bed in 8 Mixed Dormitory Room | 8.2 Very Good | | 5,976 | 1 bunk bed | Bar, Free Wifi, 24-hour front desk | | | Subway Access |
| June Six Hotel Berlin City West | Charlottenburg-Wilmersdorf, Berlin | 2.4 miles | $174 | King Room with Garden View | 8.2 Very Good | | 4,013 | 1 queen bed | Bar, Fitness center, Restaurant | | | Subway Access |
| Grimm's Potsdamer Platz | Friedrichshain-Kreuzberg, Berlin | 0.9 miles | $184 | Single Room | 8.7 Excellent | | 7,700 | 1 twin bed | Bar, Fitness center, Sauna | | | Subway Access |
| nhow Berlin | Friedrichshain-Kreuzberg, Berlin | 3.2 miles | $318 | nhow Junior Suite with River View | 8.9 Excellent | 9.3 | 5,698 | 1 king bed | Bar, Fitness center, Sustainability certification | | | Subway Access |
| The Circus Hotel | Mitte, Berlin | 1.4 miles | $211 | Medium (max 2 pax) | 8.8 Excellent | 9.4 | 773 | 1 queen bed | Bar, Fitness center, Free Wifi | | | Subway Access |
| Hotel Bleibtreu Berlin by Golden Tulip | Charlottenburg-Wilmersdorf, Berlin | 2.7 miles | $162 | Standard Room - 1 King Bed | 8.0 Very Good | | 1,958 | 1 queen bed | Free Wifi, 24-hour front desk, Baggage storage | | | |
| Michelberger Hotel | Friedrichshain-Kreuzberg, Berlin | 3.1 miles | $224 | Loft Room | 8.4 Very Good | | 2,293 | 1 twin bed | Bar, Fitness center, Restaurant | | | Subway Access |
| Schulz Hotel Berlin Wall at the East Side Gallery | Friedrichshain-Kreuzberg, Berlin | 2.4 miles | $145 | Superior Double or Twin Room | 8.6 Excellent | 9.3 | 19,789 | 1 double or 2 twins | Bar, Free Wifi, 24-hour front desk | | | |
| SANA Berlin Hotel | Charlottenburg-Wilmersdorf, Berlin | 2.1 miles | $179 | Deluxe Double Room | 8.7 Excellent | | 5,260 | 1 queen bed | Bar, Fitness center, Sauna | | | Subway Access |
| Numa Berlin Potsdamer Platz | Friedrichshain-Kreuzberg, Berlin | 0.7 miles | $144 | Double Room | 8.0 Very Good | | 655 | 1 queen bed | Fitness center, Free Wifi, Baggage storage | | | Subway Access |
| Schlosshotel Berlin | Charlottenburg-Wilmersdorf, Berlin | 5 miles | $220 | Deluxe Double Room | 8.5 Very Good | | 1,610 | Multiple bed types | Bar, Fitness center, Sauna | | | |
| Park Inn by Radisson Berlin Alexanderplatz | Mitte, Berlin | 1.6 miles | $158 | Double Room with City View | 8.1 Very Good | 9.5 | 14,894 | 1 double or 2 twins | Bar, Spa, Sustainability certification | | | Subway Access |
| Motel One Berlin-Upper West | Charlottenburg-Wilmersdorf, Berlin | 2 miles | $192 | Standard Double Room | 8.5 Very Good | 9.6 | 7,895 | 1 queen bed | Bar, Free Wifi, Sustainability certification | | | Subway Access |
| Holiday Inn Express - Berlin - Alexanderplatz by IHG | Mitte, Berlin | 1.5 miles | $170 | Standard Double Room | 8.6 Excellent | | 9,209 | 1 full bed | Bar, Free Wifi, Sustainability certification | | Yes | Subway Access |
| NH Berlin Potsdamer Platz | Friedrichshain-Kreuzberg, Berlin | 1.1 miles | $185 | Standard Double or Twin Room | 8.6 Excellent | | 1,242 | 1 double or 2 twins | Bar, Free Wifi, Sustainability certification | | | Subway Access |
| Leonardo Hotel Berlin Mitte | Mitte, Berlin | 0.6 miles | $260 | Comfort Room | 8.5 Very Good | 9.6 | 12,584 | 1 double or 2 twins | Bar, Fitness center, Sustainability certification | | | Subway Access |
| Hotel AMANO East Side | Friedrichshain-Kreuzberg, Berlin | 2.3 miles | $200 | Comfort Room | 8.4 Very Good | | 6,660 | 1 queen bed | Bar, Restaurant, Sustainability certification | | | |
| Hotel Schöneberg | Tempelhof-Schöneberg, Berlin | 2.4 miles | $132 | Single Room Hotdeal nonrefundable | 8.1 Very Good | | 700 | 1 twin bed | Bar, Free Wifi, Baggage storage | | | |
| Hotel Transit Loft | Pankow, Berlin | 2.3 miles | $95 | Double or Twin Room | 7.9 Good | | 5,762 | 1 double or 2 twins | Bar, Free Wifi, 24-hour front desk | | Yes | |
| Numa Berlin Drift | Tempelhof-Schöneberg, Berlin | 1.5 miles | $137 | Medium Room | 8.3 Very Good | | 2,821 | 1 king bed | Fitness center, Free Wifi, Baggage storage | | | Subway Access |
| Ringhotel Seehof | Charlottenburg-Wilmersdorf, Berlin | 3.7 miles | $202 | Standard Single Room | 7.9 Good | | 1,494 | 1 twin bed | Restaurant, Free Wifi, Sustainability certification | | | |
| Hampton By Hilton Berlin City East Side Gallery | Friedrichshain-Kreuzberg, Berlin | 2.9 miles | $185 | King Room | 8.6 Excellent | 9.4 | 7,553 | 1 king bed | Bar, Free Wifi, Sustainability certification | | Yes | Subway Access |
| Hotel Comenius | Friedrichshain-Kreuzberg, Berlin | 3.1 miles | $169 | Twin Room | 8.4 Very Good | | 1,664 | 2 twin beds | Free Wifi | | Yes | Subway Access |

As you can see, some fields are missing for certain entries. Let's see how we can fill in that missing information using TabPFN models.

## Step 2: Connect to Prior Labs via MCP

[Prior Labs](https://priorlabs.ai/) exposes [TabPFN](https://github.com/PriorLabs/TabPFN), a pretrained tabular foundation model, through an MCP server. We load the toolset in the Prior Labs MCP using [MCPToolset](https://docs.haystack.deepset.ai/docs/mcptoolset), which gives the agent tools for uploading datasets and running fit/predict operations.

In [19]:
from haystack_integrations.tools.mcp import MCPToolset, StreamableHttpServerInfo
from haystack.utils import Secret

# Get the server info from environment variables and create the toolset
# based on the tools exposed by the MCP server.
server_info = StreamableHttpServerInfo(
    url="https://api.priorlabs.ai/mcp/server",
    token=Secret.from_env_var("PRIOR_LABS_MCP_TOKEN"),
)
toolset = MCPToolset(server_info=server_info, eager_connect=True)

for tool in toolset.tools:
    print(f"{tool.name}: {tool.description}")

upload_dataset: Default entrypoint for file-based data.

Use this whenever the dataset is referenced as a file path, attachment, or URL.
Do not ask users to paste entire CSV contents into the conversation.

Initializes a direct upload to cloud storage. Returns a dataset_id and a
pre-signed upload_url valid for 60 minutes.

After calling this tool:
  1. HTTP PUT the file bytes directly to upload_url.
  2. Do NOT read the file contents into the conversation.
  3. Pass the returned dataset_id to fit_and_predict_from_dataset
     or predict_from_dataset.

Call this tool once per file — once for train.csv, once for test.csv.
The dataset_id has no implicit train/test meaning; label it yourself.

For ChatGPT/Claude web sessions without code execution:
  - still prefer this tool for real datasets
  - if upload cannot be executed in-session, do not paste full files inline;
    use an upload-capable MCP client/session.

Do NOT call this when the data is already a small inline array/table.

fit_a

## Step 3: Add an agent to fill in missing values

We extend the pipeline with a Haystack `Agent` that receives the structured hotel table and uses Prior Labs tools to predict missing attributes (location ratings, room sizes, transport options). Predicted values are marked in **bold** in the final output.

In [20]:
from haystack.components.agents import Agent

llm = AnthropicChatGenerator(
    model="claude-opus-4-6",
    generation_kwargs={
        "max_tokens": 10000,
    }
)

pipeline.add_component(
    "agent",
    Agent(
        chat_generator=llm,
        tools=toolset,
        user_prompt="""{% message role="user" %}
You are a data scientist working with tabular data with a specialization in travel and hospitality. You have access to a set of tools that you should use to predict the missing attributes of hotels based on the information provided in the input table.
Before you use any of the tools, make sure to prepare the hotel data appropriately, like you would do for any tabular dataset, and interpret the results to create a human-friendly table with a structure similar to the input table.
Return only a Markdown-formatted table with the same columns as the input, but with the missing attributes filled in. Do not return any text other than the table. Mark filled attributes in bold. Do not return the tool outputs directly, or your reasoning, but only interpret them and return only the final table.
{% endmessage %}""",
    ),
)

pipeline.connect("summarizer.replies", "agent.messages")

🚅 Components
  - fetcher: FirecrawlCrawler
  - prompt_builder: ChatPromptBuilder
  - summarizer: AnthropicChatGenerator
  - agent: Agent
🛤️ Connections
  - fetcher.documents -> prompt_builder.documents (list[Document])
  - prompt_builder.prompt -> summarizer.messages (list[ChatMessage])
  - summarizer.replies -> agent.messages (list[ChatMessage])

In [21]:
output = pipeline.run({
    "fetcher": {
        "urls": [
            "https://www.booking.com/searchresults.html?ss=Berlin%2C+Berlin+Federal+State%2C+Germany&efdco=1&label=gen173nr-10CAEoggI46AdIM1gEaLYBiAEBmAEzuAEHyAEM2AED6AEB-AEBiAIBqAIBuAKQgrTOBsACAdICJDllMDgyNTA1LTdlYmMtNDJkNi04NzFmLWUyYTRkNWM1NjM3ZtgCAeACAQ&aid=304142&lang=en-us&sb=1&src_elem=sb&src=index&dest_id=-1746443&dest_type=city&ac_position=0&ac_click_type=b&ac_langcode=xu&ac_suggestion_list_length=5&search_selected=true&search_pageview_id=0da2508885520c7d&ac_meta=GhAwZGEyNTA4ODg1NTIwYzdkIAAoATICeHU6BmJlcmxpbg%3D%3D&checkin=2026-05-05&checkout=2026-05-08&group_adults=1&no_rooms=1&group_children=0",
        ]
    }
})

In [22]:
# Do not display the entire response, but only extract
# the table after filling the gaps.
response = output["agent"]["last_message"].text
table = response[response.index("| Hotel Name |"):]
display(Markdown(table))

| Hotel Name | Location | Distance from Downtown | Price per Night | Room Type | User Rating | Location Rating | Number of Reviews | Beds | Amenities | Room Size | Breakfast Included | Public Transport |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| oompH Berlin 3BR Apt with Light Tones and Artwork | Friedrichshain-Kreuzberg, Berlin | 3.9 miles | $210 | Three-Bedroom Superior Apartment | 9.3 Wonderful | 9.3 | 25 | 5 beds (1 full, 1 king, 2 sofa beds, 1 queen) | Free Wifi | 1,012 ft² | | Subway Access |
| Hotel Berlin, Berlin, a member of Radisson Individuals | Mitte, Berlin | 1.3 miles | $184 | Cozy Small Room | 8.2 Very Good | **9.5** | 19,637 | 1 queen bed | Bar, Fitness center, Sustainability certification | | | Subway Access |
| Lux 11 Berlin-Mitte | Mitte, Berlin | 1.5 miles | $218 | Deluxe Room | 8.7 Excellent | 9.5 | 2,073 | 1 king bed | Bar, Restaurant, Free Wifi | | | Subway Access |
| Premier Inn Berlin Airport | Treptow-Köpenick, Berlin | 10.8 miles | $83 | Quadruple Room | 8.6 Excellent | **9.5** | 10,046 | 2 beds (1 twin, 1 full) | Restaurant, Free Wifi, 24-hour front desk | | | |
| Steigenberger Hotel Am Kanzleramt | Mitte, Berlin | 0.6 miles | $249 | Superior Plus Room | 8.8 Excellent | 9.5 | 7,654 | 1 double or 2 twins | Bar, Spa, Sustainability certification | | | Subway Access |
| Numa Berlin Boxer | Friedrichshain-Kreuzberg, Berlin | 3.8 miles | $149 | Large Room - Accessible | 8.1 Very Good | **9.4** | 1,577 | 1 king bed | Fitness center, Free Wifi | | | |
| Hollywood Media Hotel am Kurfürstendamm | Charlottenburg-Wilmersdorf, Berlin | 2.5 miles | $158 | Standard Double or Twin Room | 8.5 Very Good | 9.4 | 7,307 | 1 double or 2 twins | Bar, Fitness center, Sustainability certification | | | Subway Access |
| art'otel berlin mitte, Powered by Radisson Hotels | Mitte, Berlin | 1.4 miles | $233 | Art Room courtyard view | 8.9 Excellent | 9.3 | 3,347 | 1 double or 2 twins | Bar, Spa, Sustainability certification | | | Subway Access |
| Wilmina Hotel | Charlottenburg-Wilmersdorf, Berlin | 3.4 miles | $265 | Cozy Double Room | 9.4 Wonderful | **9.5** | 1,189 | 1 queen bed | Bar, Fitness center, Sustainability certification | | | Subway Access |
| The Mandala Suites | Mitte, Berlin | 0.6 miles | $284 | Management Suite | 9.0 Wonderful | 9.6 | 2,225 | 1 king bed | Spa, Fitness center, Sustainability certification | 732 ft² | | Subway Access |
| Townhouse Berlin | Charlottenburg-Wilmersdorf, Berlin | 2.3 miles | $187 | City Studio | 8.8 Excellent | 9.3 | 1,216 | 1 queen bed | Fitness center, Free Wifi, Sustainability certification | 280 ft² | | Subway Access |
| Numa Berlin Torstrasse | Mitte, Berlin | 1.6 miles | $189 | Medium Room | 8.4 Very Good | 9.4 | 2,737 | 1 queen bed | Fitness center, Free Wifi | | | Subway Access |
| 25hours Hotel Bikini Berlin | Charlottenburg-Wilmersdorf, Berlin | 1.8 miles | $252 | Medium Urban Twin | 8.7 Excellent | 9.5 | 2,177 | 2 twin beds | Bar, Sauna, Sustainability certification | | | Subway Access |
| NH Collection Berlin Mitte am Checkpoint Charlie | Mitte, Berlin | 0.6 miles | $258 | Premium Double or Twin Room | 8.5 Very Good | 9.4 | 8,085 | Multiple bed types | Bar, Fitness center, Sustainability certification | | | Subway Access |
| The Social Hub Berlin Alexanderplatz | Mitte, Berlin | 1.7 miles | $152 | Standard Queen Room | 8.6 Excellent | 9.3 | 9,278 | 1 queen bed | Bar, Fitness center, Restaurant | | | Subway Access |
| Hotel Augusta Am Kurfürstendamm | Charlottenburg-Wilmersdorf, Berlin | 2.3 miles | $116 | Classic Room | 8.1 Very Good | 9.4 | 2,951 | 1 full bed | Hot tub/Jacuzzi, Free Wifi, Baggage storage | | | Subway Access |
| Hotel ROMY by AMANO | Mitte, Berlin | 0.8 miles | $197 | Standard Double Room | 8.5 Very Good | 9.5 | 5,969 | 1 queen bed | Bar, Restaurant, Sustainability certification | | | Subway Access |
| Living Hotel Berlin Mitte | Mitte, Berlin | 1.4 miles | $181 | Business Double Apartment | 8.6 Excellent | **9.5** | 1,485 | 1 queen bed | Bar, Sauna, Sustainability certification | 248 ft² | | Subway Access |
| Numa Berlin Arc | Friedrichshain-Kreuzberg, Berlin | 1 miles | $130 | Twin Room | 8.3 Very Good | **9.3** | 5,269 | 2 twin beds | Fitness center, Free Wifi, Baggage storage | | | Subway Access |
| BENSIMON apartments Prenzlauer Berg | Pankow, Berlin | 2 miles | $152 | Studio Apartment with Garden View | 9.2 Wonderful | 9.5 | 597 | 2 beds (1 sofa bed, 1 queen) | Free Wifi, Baggage storage | 431 ft² | | Subway Access |
| Hotel Sachsenhof | Tempelhof-Schöneberg, Berlin | 1.6 miles | $134 | Triple Room | 8.5 Very Good | 9.3 | 5,242 | 2 beds (1 twin, 1 queen) | Free Wifi, 24-hour front desk, Baggage storage | | | Subway Access |
| InterContinental Berlin by IHG | Mitte, Berlin | 1.5 miles | $170 | Classic Room | 8.8 Excellent | **9.5** | 4,778 | 1 double or 2 twins | Bar, Spa, Sustainability certification | | | |
| martas Hotel Berlin-Mitte | Mitte, Berlin | 1 miles | $180 | Double Room | 8.8 Excellent | 9.6 | 2,619 | 1 queen bed | Free Wifi, Baggage storage | | | Subway Access |
| Orania.Berlin | Friedrichshain-Kreuzberg, Berlin | 1.9 miles | $328 | Double Room (Orania.25) | 9.4 Wonderful | **9.5** | 536 | 1 queen bed | Bar, Fitness center, Restaurant | | | Subway Access |
| Pestana Berlin Tiergarten | Mitte, Berlin | 1.3 miles | $184 | Deluxe Premium Room | 8.4 Very Good | **9.5** | 5,473 | 1 king bed | Bar, Fitness center, Sauna | | | |
| Park Plaza Berlin | Charlottenburg-Wilmersdorf, Berlin | 2.5 miles | $239 | Superior Double Room | 8.3 Very Good | **9.5** | 5,253 | 1 queen bed | Bar, Fitness center, Sustainability certification | | | Subway Access |
| Holiday Inn Berlin City East Side by IHG | Friedrichshain-Kreuzberg, Berlin | 3 miles | $199 | Standard Twin Room with Courtyard View | 8.7 Excellent | 9.5 | 5,239 | 2 twin beds | Bar, Fitness center, Restaurant | | | Subway Access |
| Myer's Hotel Berlin | Pankow, Berlin | 2 miles | $256 | Comfort Double or Twin Room | 9.0 Wonderful | **9.5** | 997 | 1 double or 2 twins | Bar, Spa, Fitness center | | Yes | Subway Access |
| Dorint Kurfürstendamm Berlin | Charlottenburg-Wilmersdorf, Berlin | 2.1 miles | $259 | Comfort Room with Kingsize Bed | 8.8 Excellent | 9.5 | 4,691 | 1 king bed | Bar, Fitness center, Sustainability certification | | | Subway Access |
| The Ritz-Carlton, Berlin | Mitte, Berlin | 0.4 miles | $433 | Deluxe King Room | 9.3 Wonderful | 9.6 | 1,097 | 1 king bed | Bar, Fitness center, Sustainability certification | | | Subway Access |
| Radisson RED Berlin Kudamm | Charlottenburg-Wilmersdorf, Berlin | 2.2 miles | $242 | Standard Single Room | 8.6 Excellent | **9.4** | 2,427 | 1 twin bed | Fitness center, Restaurant, Sustainability certification | | | Subway Access |
| IntercityHotel Berlin Hauptbahnhof | Mitte, Berlin | 0.7 miles | $244 | Business Double Room | 8.0 Very Good | 9.4 | 10,484 | 1 queen bed | Bar, Free Wifi, Sustainability certification | | | Subway Access |
| The Mandala Berlin, a Member of Design Hotels | Mitte, Berlin | 0.5 miles | $313 | Garden Studio with Balcony | 9.1 Wonderful | 9.5 | 1,746 | 1 queen bed | Bar, Spa, Fitness center | | | Subway Access |
| Telegraphenamt | Mitte, Berlin | 0.9 miles | $454 | Deluxe Double Room | 8.8 Excellent | 9.7 | 720 | 1 king bed | Bar, Fitness center, Sustainability certification | | | |
| Arbio I Luxury Apartments in East Side Gallery | Friedrichshain-Kreuzberg, Berlin | 2.8 miles | $161 | Comfort Studio | 8.5 Very Good | 9.4 | 1,004 | 2 beds (1 full, 1 sofa bed) | Free Wifi, Baggage storage | 344 ft² | | Subway Access |
| Hotel Gat Point Charlie | Mitte, Berlin | 0.7 miles | $128 | Standard Double Room | 8.5 Very Good | 9.5 | 7,548 | 1 double or 2 twins | Bar, Free Wifi, Sustainability certification | | | Subway Access |
| Berlin Marriott Hotel | Mitte, Berlin | 0.4 miles | $302 | Deluxe Room, 1 King | 8.6 Excellent | 9.5 | 3,141 | 1 king bed | Bar, Fitness center, Sustainability certification | | | Subway Access |
| Hotel Zoo Berlin | Charlottenburg-Wilmersdorf, Berlin | 2.2 miles | $238 | Superior Room | 8.8 Excellent | 9.6 | 3,205 | 1 double or 2 twins | Bar, Fitness center, Sustainability certification | | | Subway Access |
| Hotel AMANO Grand Central | Mitte, Berlin | 0.9 miles | $229 | COMFY Plus | 8.2 Very Good | 9.3 | 10,475 | 1 double or 2 twins | Bar, Restaurant, Sustainability certification | | | Subway Access |
| Art Apartments | Mitte, Berlin | 1.1 miles | $160 | Comfort Triple Room with Shower | 7.2 Good | **9.5** | 1,724 | 3 king beds | Free Wifi | | | Subway Access |
| Hotel AMO by AMANO Friedrichstraße | Mitte, Berlin | 0.8 miles | $215 | COMFY Room | 8.3 Very Good | 9.4 | 4,562 | 1 double or 2 twins | Bar, Restaurant, Sustainability certification | | | Subway Access |
| Melarose Feng Shui Hotel | Pankow, Berlin | 2.6 miles | $146 | Single Room | 8.3 Very Good | **9.4** | 1,478 | 1 twin bed | Free Wifi, Baggage storage | | | |
| Hotel Nikolai Residence | Mitte, Berlin | 1.3 miles | $183 | Double Room | 8.9 Excellent | 9.7 | 2,562 | 1 double or 2 twins | Free Wifi, Baggage storage | | | Subway Access |
| The Hoxton, Berlin | Charlottenburg-Wilmersdorf, Berlin | 2.3 miles | $233 | Standard Twin Room | 8.6 Excellent | **9.4** | 2,385 | 2 twin beds | Bar, Free Wifi, Sustainability certification | | | Subway Access |
| Motel One Berlin-Alexanderplatz | Mitte, Berlin | 1.5 miles | $158 | Standard Double Room | 8.7 Excellent | 9.6 | 16,191 | 1 queen bed | Bar, Free Wifi, Sustainability certification | | | Subway Access |
| Scandic Berlin Potsdamer Platz | Mitte, Berlin | 0.8 miles | $198 | Standard Triple Room | 8.4 Very Good | 9.3 | 11,774 | 2 beds (1 sofa bed, 1 queen) | Bar, Fitness center, Sustainability certification | | | Subway Access |
| Lindner Hotel Berlin Ku'damm, part of JdV by Hyatt | Charlottenburg-Wilmersdorf, Berlin | 2.2 miles | $182 | Queen Room - High Floor | 8.4 Very Good | 9.6 | 2,175 | 1 queen bed | Restaurant, Free Wifi, 24-hour front desk | | | Subway Access |
| Hotel MANI by AMANO | Mitte, Berlin | 1.3 miles | $179 | Double Room | 8.3 Very Good | 9.3 | 2,430 | 1 double or 2 twins | Bar, Restaurant, Sustainability certification | | | Subway Access |
| Hotel Q! Berlin | Charlottenburg-Wilmersdorf, Berlin | 2.5 miles | $171 | Single Room | 8.7 Excellent | 9.5 | 1,196 | 1 twin bed | Fitness center, Sauna, Free Wifi | | | Subway Access |
| Hotel Adlon Kempinski Berlin | Mitte, Berlin | 0.1 miles | $417 | Executive Room King | 9.1 Wonderful | 9.8 | 3,891 | 1 king bed | Bar, Fitness center, Hot tub/Jacuzzi | | | Subway Access |
| Maritim proArte Hotel Berlin | Mitte, Berlin | 0.5 miles | $291 | Classic Double Room | 8.5 Very Good | 9.7 | 11,379 | 1 double or 2 twins | Bar, Spa, Sustainability certification | | | Subway Access |
| aletto Hotel Potsdamer Platz | Friedrichshain-Kreuzberg, Berlin | 1.1 miles | $121 | Double or Twin Room | 8.8 Excellent | **9.5** | 8,842 | 2 twin beds | Bar, Restaurant, Sustainability certification | | | Subway Access |
| Holiday Inn Express Berlin City Centre by IHG | Friedrichshain-Kreuzberg, Berlin | 1.1 miles | $173 | Standard Room | 8.5 Very Good | **9.5** | 9,791 | 1 double or 2 twins | Bar, Free Wifi, Sustainability certification | | Yes | Subway Access |
| DoubleTree by Hilton Berlin Ku'damm | Charlottenburg-Wilmersdorf, Berlin | 2 miles | $252 | Twin Room | 8.7 Excellent | 9.4 | 3,942 | 2 twin beds | Bar, Fitness center, Sustainability certification | | | Subway Access |
| Industriepalast Berlin | Friedrichshain-Kreuzberg, Berlin | 3.1 miles | $24 | Single Bed in 8 Mixed Dormitory Room | 8.2 Very Good | **9.4** | 5,976 | 1 bunk bed | Bar, Free Wifi, 24-hour front desk | | | Subway Access |
| June Six Hotel Berlin City West | Charlottenburg-Wilmersdorf, Berlin | 2.4 miles | $174 | King Room with Garden View | 8.2 Very Good | **9.4** | 4,013 | 1 queen bed | Bar, Fitness center, Restaurant | | | Subway Access |
| Grimm's Potsdamer Platz | Friedrichshain-Kreuzberg, Berlin | 0.9 miles | $184 | Single Room | 8.7 Excellent | **9.5** | 7,700 | 1 twin bed | Bar, Fitness center, Sauna | | | Subway Access |
| nhow Berlin | Friedrichshain-Kreuzberg, Berlin | 3.2 miles | $318 | nhow Junior Suite with River View | 8.9 Excellent | 9.3 | 5,698 | 1 king bed | Bar, Fitness center, Sustainability certification | | | Subway Access |
| The Circus Hotel | Mitte, Berlin | 1.4 miles | $211 | Medium (max 2 pax) | 8.8 Excellent | 9.4 | 773 | 1 queen bed | Bar, Fitness center, Free Wifi | | | Subway Access |
| Hotel Bleibtreu Berlin by Golden Tulip | Charlottenburg-Wilmersdorf, Berlin | 2.7 miles | $162 | Standard Room - 1 King Bed | 8.0 Very Good | **9.4** | 1,958 | 1 queen bed | Free Wifi, 24-hour front desk, Baggage storage | | | |
| Michelberger Hotel | Friedrichshain-Kreuzberg, Berlin | 3.1 miles | $224 | Loft Room | 8.4 Very Good | **9.4** | 2,293 | 1 twin bed | Bar, Fitness center, Restaurant | | | Subway Access |
| Schulz Hotel Berlin Wall at the East Side Gallery | Friedrichshain-Kreuzberg, Berlin | 2.4 miles | $145 | Superior Double or Twin Room | 8.6 Excellent | 9.3 | 19,789 | 1 double or 2 twins | Bar, Free Wifi, 24-hour front desk | | | |
| SANA Berlin Hotel | Charlottenburg-Wilmersdorf, Berlin | 2.1 miles | $179 | Deluxe Double Room | 8.7 Excellent | **9.5** | 5,260 | 1 queen bed | Bar, Fitness center, Sauna | | | Subway Access |
| Numa Berlin Potsdamer Platz | Friedrichshain-Kreuzberg, Berlin | 0.7 miles | $144 | Double Room | 8.0 Very Good | **9.5** | 655 | 1 queen bed | Fitness center, Free Wifi, Baggage storage | | | Subway Access |
| Schlosshotel Berlin | Charlottenburg-Wilmersdorf, Berlin | 5 miles | $220 | Deluxe Double Room | 8.5 Very Good | **9.4** | 1,610 | Multiple bed types | Bar, Fitness center, Sauna | | | |
| Park Inn by Radisson Berlin Alexanderplatz | Mitte, Berlin | 1.6 miles | $158 | Double Room with City View | 8.1 Very Good | 9.5 | 14,894 | 1 double or 2 twins | Bar, Spa, Sustainability certification | | | Subway Access |
| Motel One Berlin-Upper West | Charlottenburg-Wilmersdorf, Berlin | 2 miles | $192 | Standard Double Room | 8.5 Very Good | 9.6 | 7,895 | 1 queen bed | Bar, Free Wifi, Sustainability certification | | | Subway Access |
| Holiday Inn Express - Berlin - Alexanderplatz by IHG | Mitte, Berlin | 1.5 miles | $170 | Standard Double Room | 8.6 Excellent | **9.3** | 9,209 | 1 full bed | Bar, Free Wifi, Sustainability certification | Yes | | Subway Access |
| NH Berlin Potsdamer Platz | Friedrichshain-Kreuzberg, Berlin | 1.1 miles | $185 | Standard Double or Twin Room | 8.6 Excellent | **9.5** | 1,242 | 1 double or 2 twins | Bar, Free Wifi, Sustainability certification | | | Subway Access |
| Leonardo Hotel Berlin Mitte | Mitte, Berlin | 0.6 miles | $260 | Comfort Room | 8.5 Very Good | 9.6 | 12,584 | 1 double or 2 twins | Bar, Fitness center, Sustainability certification | | | Subway Access |
| Hotel AMANO East Side | Friedrichshain-Kreuzberg, Berlin | 2.3 miles | $200 | Comfort Room | 8.4 Very Good | **9.5** | 6,660 | 1 queen bed | Bar, Restaurant, Sustainability certification | | | |
| Hotel Schöneberg | Tempelhof-Schöneberg, Berlin | 2.4 miles | $132 | Single Room Hotdeal nonrefundable | 8.1 Very Good | **9.4** | 700 | 1 twin bed | Bar, Free Wifi, Baggage storage | | | |
| Hotel Transit Loft | Pankow, Berlin | 2.3 miles | $95 | Double or Twin Room | 7.9 Good | **9.4** | 5,762 | 1 double or 2 twins | Bar, Free Wifi, 24-hour front desk | | Yes | |
| Numa Berlin Drift | Tempelhof-Schöneberg, Berlin | 1.5 miles | $137 | Medium Room | 8.3 Very Good | **9.3** | 2,821 | 1 king bed | Fitness center, Free Wifi, Baggage storage | | | Subway Access |
| Ringhotel Seehof | Charlottenburg-Wilmersdorf, Berlin | 3.7 miles | $202 | Standard Single Room | 7.9 Good | **9.4** | 1,494 | 1 twin bed | Restaurant, Free Wifi, Sustainability certification | | | |
| Hampton By Hilton Berlin City East Side Gallery | Friedrichshain-Kreuzberg, Berlin | 2.9 miles | $185 | King Room | 8.6 Excellent | 9.4 | 7,553 | 1 king bed | Bar, Free Wifi, Sustainability certification | | Yes | Subway Access |
| Hotel Comenius | Friedrichshain-Kreuzberg, Berlin | 3.1 miles | $169 | Twin Room | 8.4 Very Good | **9.4** | 1,664 | 2 twin beds | Free Wifi | | Yes | Subway Access |

## Conclusion

LLMs are remarkably good at turning messy, unstructured web content into clean, structured tables, but they struggle with numerical reasoning and will often leave gaps where data wasn't explicitly present on the page. This is where classical tabular ML still shines: given a few known examples, TabPFN can extrapolate missing values with no fine-tuning required.

The combination is powerful: use an LLM to handle the unstructured-to-structured step, then hand off to a tabular model for the gaps. Haystack's `Agent` with MCP tools makes it straightforward to wire these two worlds together in a single pipeline.